In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
import torch
from bridgit import Bridgit, Player, Visualizer
from bridgit.schema import Move
from bridgit.ai import BridgitNet, NetWrapper, MCTS
from bridgit.config import BoardConfig, MCTSConfig, NeuralNetConfig
from bridgit import schema
from plotly.subplots import make_subplots

In [ ]:
board_config = BoardConfig(size=5)
mcts_config = MCTSConfig(num_simulations=5000)
net_config = NeuralNetConfig(num_channels=32, num_res_blocks=2)

net = BridgitNet(board_config, net_config)
wrapper = NetWrapper(net)
mcts = MCTS(wrapper, mcts_config)

game = Bridgit(board_config)

g = board_config.grid_size
print(f"Board: {g}x{g}")
print(f"Legal crossings: {len(game.get_available_moves())}")
print(f"Device: {wrapper.device}")

In [4]:
game = Bridgit(board_config)
game.make_move(schema.Move(row=1, col=1))
game.make_move(schema.Move(row=2, col=2))
game.make_move(schema.Move(row=3, col=3))

# Get raw net prediction
policy, value = mcts._predict(game)

# Mask to only legal crossings
mask = game.to_mask()
masked_policy = policy * mask
masked_policy /= masked_policy.sum()

fig = make_subplots(rows=1, cols=2, subplot_titles=["Raw Policy (all cells)", "Masked Policy (legal moves only)"])

fig.add_trace(Visualizer.visualize_array(policy, colorscale="Blues").data[0], row=1, col=1)
fig.add_trace(Visualizer.visualize_array(masked_policy, colorscale="Reds").data[0], row=1, col=2)

fig.update_layout(width=700, height=350, title=f"Net value estimate: {value:.3f}")
fig.show()

In [ ]:
game = Bridgit(board_config)
game.make_move(schema.Move(row=1, col=1))
game.make_move(schema.Move(row=2, col=2))
game.make_move(schema.Move(row=3, col=3))

# MCTS search
visit_counts = mcts._search(game).visit_counts()
mcts_probs = mcts.get_action_probs(game, temperature=1.0)

fig = make_subplots(rows=1, cols=2, subplot_titles=["Visit Counts", "MCTS Policy (temp=1)"])

fig.add_trace(Visualizer.visualize_array(visit_counts, colorscale="Greens").data[0], row=1, col=1)
fig.add_trace(Visualizer.visualize_array(mcts_probs, colorscale="Reds").data[0], row=1, col=2)

fig.update_layout(width=700, height=350, title="MCTS concentrates visits on promising moves")
fig.show()

In [ ]:
mcts_probs

tensor([[0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0246, 0.0000, 0.0198, 0.0000, 0.0212, 0.0000,
         0.0322, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0228, 0.0000, 0.0212, 0.0000, 0.0208,
         0.0000, 0.0000],
        [0.0000, 0.0214, 0.0000, 0.0000, 0.0000, 0.0198, 0.0000, 0.0224, 0.0000,
         0.0280, 0.0000],
        [0.0000, 0.0000, 0.0508, 0.0000, 0.0318, 0.0000, 0.0200, 0.0000, 0.0196,
         0.0000, 0.0000],
        [0.0000, 0.0320, 0.0000, 0.0224, 0.0000, 0.0362, 0.0000, 0.0206, 0.0000,
         0.0234, 0.0000],
        [0.0000, 0.0000, 0.0238, 0.0000, 0.0228, 0.0000, 0.0288, 0.0000, 0.0336,
         0.0000, 0.0000],
        [0.0000, 0.0410, 0.0000, 0.0270, 0.0000, 0.0312, 0.0000, 0.0284, 0.0000,
         0.0212, 0.0000],
        [0.0000, 0.0000, 0.0212, 0.0000, 0.0384, 0.0000, 0.0324, 0.0000, 0.0172,
         0.0000, 0.0000],
        [0.0000, 0.0292, 0.0000, 0.02